In [1]:
import os
import random
from itertools import product
from time import perf_counter

import numpy as np
import polars as pl
from aeon.classification.base import BaseClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
)
from aeon.transformations.collection.convolution_based import Rocket
from aeon.datasets.tsc_datasets import univariate
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
)
from aeon.classification.convolution_based import RocketClassifier
from sklearn.metrics import accuracy_score
from aeon.classification.interval_based import QUANTClassifier
from autotsc import utils, models
from aeon.classification.convolution_based import MultiRocketHydraClassifier
from tqdm import tqdm

2025-12-02 14:40:10.292840: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
write_dir = "experiments/automl_val_vs_test_accuracy_correct"
os.makedirs(write_dir, exist_ok=True)

In [3]:
datasets = list(univariate)
# random.shuffle(datasets)

In [4]:
def get_model(model_name: str, seed: int):
    if model_name == "rocket":
        model = RocketClassifier(
            n_jobs=-1,
            random_state=seed,
            estimator=models.RidgeClassifierCVWithProba(alphas=np.logspace(-3, 3, 10))
        )
    elif model_name == "catch22":
        model = Catch22Classifier(random_state=seed, n_jobs=-1)
    elif model_name == "quant":
        model = QUANTClassifier(random_state=seed)
    elif model_name == "hydra":
        from aeon.classification.convolution_based import HydraClassifier
        model = HydraClassifier(random_state=seed, n_jobs=-1)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model

In [ ]:
def train_model_n_times(model, X_train, y_train, n_times: int):
    models = []
    for _ in range(n_times):
        cloned_model = clone(model)
        cloned_model.fit(X_train, y_train)
        models.append(cloned_model)
    return models

In [ ]:
for dataset in tqdm(datasets[:1]):
    for model_name in ['rocket']:#, 'catch22', 'quant', 'hydra']:
        for run in range(1):
            n_folds = 10

            stats = {
                "dataset": dataset,
                "model": model_name,
                "run": run,
            }

            hash_val = pl.DataFrame([stats]).hash_rows(seed=42, seed_1=1, seed_2=2, seed_3=3).item()
            file = f"{write_dir}/{hash_val}.parquet"

            if os.path.exists(file):
                print(f"Skipping: Dataset={dataset}, Run={run}, Model={model_name}")
                continue

            model = get_model(model_name, seed=run)
            X_train, y_train, X_test, y_test = utils.load_dataset(dataset)

            
            model.fit(X_train, y_train)
            test_preds = model.predict(X_test)
            test_pred_probs = model.predict_proba(X_test)
            test_acc = accuracy_score(y_test, test_preds)
            
            folds = utils.get_folds(X_train, y_train, n_splits=min(n_folds, len(X_train)), random_state=run)

            preds = []
            true = []
            pred_probas = []
            for train_idx, val_idx in folds:
                X_tr, y_tr = X_train[train_idx], y_train[train_idx]
                X_val, y_val = X_train[val_idx], y_train[val_idx]
                
                fold_model = get_model(model_name, seed=run)
                fold_model.fit(X_tr, y_tr)
                val_preds = fold_model.predict(X_val)
                val_pred_probs = fold_model.predict_proba(X_val)

                preds.extend(zip(val_idx, val_preds))
                true.extend(zip(val_idx, y_val))
                pred_probas.extend(zip(val_idx, val_pred_probs))

            preds = sorted(preds, key=lambda x: x[0])
            true = sorted(true, key=lambda x: x[0])
            pred_probas = sorted(pred_probas, key=lambda x: x[0])
            preds = [x[1].item() for x in preds]
            true = [x[1].item() for x in true]
            pred_probas = [x[1].tolist() for x in pred_probas]


            vall_acc = accuracy_score(true, preds)

            stats["test_acc"] = test_acc
            stats["val_acc"] = vall_acc
            stats["test_true"] = list(y_test)
            stats["val_true"] = list(true)
            stats["test_preds"] = list(test_preds)
            stats["val_preds"] = list(preds)
            stats["test_probabilities"] = list(test_pred_probs)
            stats["val_probabilities"] = list(pred_probas)
            
            df_stat = pl.DataFrame([stats])
            df_stat.write_parquet(file, mkdir=True)

            print(f"Saved: Dataset={dataset}, Run={run}, Model={model_name}")

  0%|          | 0/1 [00:00<?, ?it/s]

StratifiedKFold failed, falling back to regular KFold with n_splits=20
Saved: Dataset=ACSF1, Run=0, Model=rocket
StratifiedKFold failed, falling back to regular KFold with n_splits=20
Saved: Dataset=ACSF1, Run=0, Model=catch22
StratifiedKFold failed, falling back to regular KFold with n_splits=20
Saved: Dataset=ACSF1, Run=0, Model=quant
StratifiedKFold failed, falling back to regular KFold with n_splits=20


100%|██████████| 1/1 [03:42<00:00, 222.76s/it]

Saved: Dataset=ACSF1, Run=0, Model=hydra


In [6]:
df = pl.read_parquet(f"{write_dir}/*.parquet").sort('dataset')
df

dataset,model,run,test_acc,val_acc,test_true,val_true,test_preds,val_preds,test_probabilities,val_probabilities
str,str,i64,f64,f64,list[str],list[str],list[str],list[str],list[list[f64]],list[list[f64]]
"""ACSF1""","""hydra""",0,0.85,0.75,"[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""4""]","[[0.0, 0.0, … 1.0], [0.0, 0.0, … 1.0], … [0.0, 1.0, … 0.0]]","[[0.0, 0.0, … 1.0], [0.0, 0.0, … 1.0], … [0.0, 0.0, … 0.0]]"
"""ACSF1""","""catch22""",0,0.85,0.65,"[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""4""]","[""9"", ""9"", … ""4""]","[[0.01, 0.0, … 0.86], [0.005, 0.01, … 0.63], … [0.015, 0.345, … 0.0]]","[[0.0, 0.005, … 0.895], [0.0, 0.005, … 0.775], … [0.12, 0.23, … 0.0]]"
"""ACSF1""","""rocket""",0,0.88,0.73,"[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""1""]","[[0.091763, 0.08726, … 0.186603], [0.085799, 0.084541, … 0.180325], … [0.102551, 0.238874, … 0.087064]]","[[0.08852, 0.084913, … 0.193816], [0.093635, 0.087222, … 0.179326], … [0.127403, 0.171202, … 0.070897]]"
"""ACSF1""","""quant""",0,0.91,0.68,"[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""1""]","[""9"", ""9"", … ""4""]","[[0.0, 0.0, … 0.97], [0.0, 0.0, … 0.935], … [0.02, 0.715, … 0.0]]","[[0.0, 0.0, … 0.99], [0.0, 0.0, … 0.9], … [0.085, 0.37, … 0.0]]"
